# TrueTone: MUStARD++ final project

PID-02 | Bennett University, Greater Noida.

The final app is fitted on 1,202 balanced MUStARD++ targets (601 per class). MUStARD (690 targets) is retained only as a baseline. The matched comparison tests the same 688 shared targets, with grouped outer and inner splits, three repetitions of five folds, and separate held-out-show tests. All encoders are frozen; classifiers are logistic regression. No gated network is used in this final study.

Install `requirements-ui.txt` in Python 3.12 first. Run from the repository root. Data downloads can be several GB. The original 690-target notebook is archived under `research/baseline/`.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
import numpy as np, pandas as pd, torch, soundfile as sf, librosa
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoFeatureExtractor, AutoModelForAudioClassification
ROOT = Path.cwd()
assert (ROOT / 'inference.py').exists(), 'Run from the TrueTone repository root.'
os.environ.setdefault('TRUETONE_DIR', str(ROOT / 'truetone'))
os.environ.setdefault('HF_HOME', str(ROOT / 'truetone/hf'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE = ROOT / 'truetone/pp'
CACHE.mkdir(parents=True, exist_ok=True)
CACHE_KEY = 'live'
def run(script):
    subprocess.run([sys.executable, str(ROOT / script)], cwd=ROOT, check=True)

## Frozen feature functions

These definitions are shared with the local app through a verified AST signature. They are identical to those used to generate the trained weights. Calling them loads encoders; merely defining them does not. Ground-truth emotion annotations are never used as prediction inputs.

In [ ]:
@torch.no_grad()
def embed_text(texts, name, pair=None, bs=32, max_len=256):
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModel.from_pretrained(name).to(DEVICE).eval()
    if DEVICE == 'cuda':
        m = m.half()
    out = []
    for i in tqdm(range(0, len(texts), bs), desc=f'embed {name}'):
        a = texts[i:i + bs]
        b = pair[i:i + bs] if pair is not None else None
        enc = tok(a, b, padding=True, truncation='only_second' if b is not None else True, max_length=max_len, return_tensors='pt').to(DEVICE)
        h = m(**enc).last_hidden_state.float()
        mask = enc['attention_mask'].unsqueeze(-1).float()
        out.append(((h * mask).sum(1) / mask.sum(1)).cpu().numpy())
    del m
    torch.cuda.empty_cache()
    return np.concatenate(out).astype(np.float32)

@torch.no_grad()
def text_sentiment(texts, name, bs=64):
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
    id2l = {int(k): v.lower() for k, v in m.config.id2label.items()}
    print('text-sentiment labels:', id2l)
    find = lambda pre, dflt: next((i for i, l in id2l.items() if l.startswith(pre)), dflt)
    order = [find('neg', 0), find('neu', 1), find('pos', 2)]
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
        out.append(torch.softmax(m(**enc).logits.float(), -1)[:, order].cpu().numpy())
    del m
    torch.cuda.empty_cache()
    return np.concatenate(out).astype(np.float32)

def load_wav(p, max_s=30):
    y, sr = sf.read(p, dtype='float32')
    if y.ndim > 1:
        y = y.mean(1)
    assert sr == 16000
    y = y[:sr * max_s]
    return np.pad(y, (0, max(0, 8000 - len(y))))

@torch.no_grad()
def wavlm_embed(paths, name):
    fe = AutoFeatureExtractor.from_pretrained(name)
    m = AutoModel.from_pretrained(name).to(DEVICE).eval()
    out = []
    for p in tqdm(paths, desc=f'embed {name}'):
        x = fe(load_wav(p), sampling_rate=16000, return_tensors='pt')['input_values'].to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            hs = m(x, output_hidden_states=True).hidden_states
        hs = torch.stack(hs, 0).float()[:, 0]
        L = hs.shape[0] - 1
        mid = hs[L // 3:2 * L // 3 + 1].mean(0)
        out.append(torch.cat([mid.mean(0), mid.std(0)]).cpu().numpy())
    del m
    torch.cuda.empty_cache()
    return np.nan_to_num(np.stack(out)).astype(np.float32)

@torch.no_grad()
def speech_emotion(paths, name):
    fe = AutoFeatureExtractor.from_pretrained(name)
    m = AutoModelForAudioClassification.from_pretrained(name).to(DEVICE).eval()
    labels = [m.config.id2label[i] for i in range(m.config.num_labels)]
    out = []
    for p in tqdm(paths, desc=f'emotion {name}'):
        x = fe(load_wav(p), sampling_rate=16000, return_tensors='pt')['input_values'].to(DEVICE)
        out.append(torch.softmax(m(x).logits.float(), -1)[0].cpu().numpy())
    del m
    torch.cuda.empty_cache()
    json.dump(labels, open(CACHE / f'emo_labels_{CACHE_KEY}.json', 'w'))
    return np.stack(out).astype(np.float32)

def prosody_feats(path, n_words, sr=16000, hop=256, frame=1024):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y = y[:sr * 30]
    if len(y) < sr // 4:
        y = np.pad(y, (0, sr // 4 - len(y)))
    dur = len(y) / sr
    F = {}
    f0, vflag, _ = librosa.pyin(y, fmin=65, fmax=500, sr=sr, frame_length=frame, hop_length=hop)
    idx = np.where(~np.isnan(f0))[0]
    v = f0[idx]
    F['f0_voiced_frac'] = len(idx) / max(len(f0), 1)
    if len(v) >= 5:
        lf = np.log2(v)
        t = idx * hop / sr
        F.update(f0_mean=v.mean(), f0_p10=np.percentile(v, 10), f0_p90=np.percentile(v, 90), f0_std_st=12 * lf.std(), f0_range_st=12 * (np.percentile(lf, 95) - np.percentile(lf, 5)), f0_slope_st_s=12 * np.polyfit(t, lf, 1)[0], jitter=float(np.mean(np.abs(np.diff(v))[np.diff(idx) == 1] / v[:-1][np.diff(idx) == 1])) if np.any(np.diff(idx) == 1) else 0.0)
    else:
        F.update(f0_mean=0, f0_p10=0, f0_p90=0, f0_std_st=0, f0_range_st=0, f0_slope_st_s=0, jitter=0)
    rms = librosa.feature.rms(y=y, frame_length=frame, hop_length=hop)[0]
    db = 20 * np.log10(rms + 1e-06)
    tt = np.arange(len(db)) * hop / sr
    F.update(rms_db_mean=db.mean(), rms_db_std=db.std(), rms_db_max=db.max(), rms_dyn_range=np.percentile(db, 90) - np.percentile(db, 10), rms_slope=np.polyfit(tt, db, 1)[0] if len(db) > 2 else 0, shimmer=np.mean(np.abs(np.diff(db))))
    iv = librosa.effects.split(y, top_db=30, frame_length=frame, hop_length=hop)
    speech_dur = sum((e - s for s, e in iv)) / sr
    gaps = np.array([(iv[i + 1][0] - iv[i][1]) / sr for i in range(len(iv) - 1)]) if len(iv) > 1 else np.array([])
    gaps = gaps[gaps >= 0.15]
    onsets = librosa.onset.onset_detect(y=y, sr=sr, hop_length=hop, units='time')
    F.update(duration=dur, speech_dur=speech_dur, speech_ratio=speech_dur / dur, n_pauses=len(gaps), pause_total=gaps.sum() if len(gaps) else 0, pause_ratio=gaps.sum() / dur if len(gaps) else 0, pause_mean=gaps.mean() if len(gaps) else 0, pause_max=gaps.max() if len(gaps) else 0, onset_rate=len(onsets) / dur, onset_rate_speech=len(onsets) / max(speech_dur, 0.1), words_per_sec=n_words / dur, words_per_sec_speech=n_words / max(speech_dur, 0.1), articulation_rate=n_words / max(speech_dur, 0.1))
    S = np.abs(librosa.stft(y, n_fft=frame, hop_length=hop))
    cen = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    F.update(cent_mean=cen.mean(), cent_std=cen.std(), flat_mean=librosa.feature.spectral_flatness(S=S).mean(), zcr_mean=librosa.feature.zero_crossing_rate(y, frame_length=frame, hop_length=hop).mean(), rolloff_mean=librosa.feature.spectral_rolloff(S=S, sr=sr).mean())
    mf = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=frame, hop_length=hop)
    for k in range(13):
        F[f'mfcc{k}_mean'] = mf[k].mean()
        F[f'mfcc{k}_std'] = mf[k].std()
    return {k: float(np.nan_to_num(x)) for k, x in F.items()}

def emo_valence(lbl):
    l = lbl.lower()
    if any((k in l for k in ('hap', 'joy', 'surpris', 'exc'))):
        return 1.0
    if any((k in l for k in ('ang', 'sad', 'fear', 'disg', 'fru'))):
        return -1.0
    return 0.0

## Download official MUStARD++ annotations and verify mirror archives

In [ ]:
run('experiment/download.py')

## Prepare original MUStARD features for the controlled baseline

In [ ]:
run('experiment/prepare_baseline.py')

## Prepare MUStARD++ target audio and frozen features

In [ ]:
run('experiment/prepare.py')

## Run paired evaluation and refit the final MUStARD++ app weights

In [ ]:
run('experiment/compare.py')

## Read the measured results

Metrics below use averaged out-of-fold probabilities on the same 688 targets, not training accuracy. Outer-fold scalers are fitted only on outer training data and reused during inner C selection. Exact duplicate grouping does not eliminate all related dialogue/speaker effects. The 95% paired group-bootstrap interval includes zero for the multimodal dataset difference. Report the observed gain without claiming statistical proof or live microphone accuracy. The separate final refit uses all 1,202 targets and is for serving, not evaluation.

In [ ]:
display(pd.read_csv(ROOT / 'results/mustardpp/summary.csv'))
display(pd.read_csv(ROOT / 'results/mustardpp/paired_bootstrap.csv'))
display(pd.read_csv(ROOT / 'results/mustardpp/held_out_show.csv'))